# Brain Tumor Classification: train with a Colab GPU

This notebook runs the repository pipeline on Google Colab. It prepares the Figshare data, trains the CNN and ViT models one at a time, evaluates on a held-out split, and compares models using validation and test metrics.

**Results status:** The README records three CNN validation selection scores from an earlier run. The Colab runtime ended before the ViT runs; this notebook does not fill in or assume ViT results. Re-run models to create your own checkpoints and results. Validation selection scores are not test accuracy.

Enable a GPU via **Runtime > Change runtime type > GPU** (or another available accelerator). For the four ViTs and especially ViT-L, a high memory GPU is helpful. This notebook saves data and checkpoints under Google Drive to preserve completed work across runtime resets.

The original notebook had a leakage risk because it augmented images before splitting. This runbook uses the refactored scripts, which split source samples first and apply augmentation only to training samples. See the repository README for methodology and references.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
STUDY_DIR = Path('/content/drive/MyDrive/brain_tumor_study')
RAW_DIR = STUDY_DIR / 'data' / 'raw'
PROCESSED_DIR = STUDY_DIR / 'data' / 'processed'
CHECKPOINT_DIR = STUDY_DIR / 'checkpoints'
REPORT_DIR = STUDY_DIR / 'reports'
for path in (RAW_DIR, PROCESSED_DIR, CHECKPOINT_DIR, REPORT_DIR):
    path.mkdir(parents=True, exist_ok=True)
print('Persistent study directory:', STUDY_DIR)

## 1. Get the project code

If the repo already exists in this Colab runtime, this cell pulls the latest version. Otherwise it clones the public repository.

In [ ]:
!git clone https://github.com/OUTLAOUAIT/Deep-Hybrid-Learning-for-Brain-Tumor-Classification.git
%cd /content/Deep-Hybrid-Learning-for-Brain-Tumor-Classification

from pathlib import Path
REPO = Path('/content/Deep-Hybrid-Learning-for-Brain-Tumor-Classification')
# Persist checkpoints, histories, reports, figures, source archives and prepared data on Drive.
for local_dir, drive_dir in [
    (REPO / 'outputs' / 'checkpoints', CHECKPOINT_DIR),
    (REPO / 'outputs' / 'reports', REPORT_DIR),
    (REPO / 'outputs' / 'figures', STUDY_DIR / 'figures'),
    (REPO / 'data' / 'raw', RAW_DIR),
    (REPO / 'data' / 'processed', PROCESSED_DIR),
]:
    local_dir.parent.mkdir(parents=True, exist_ok=True)
    drive_dir.mkdir(parents=True, exist_ok=True)
    if local_dir.is_symlink():
        continue
    if local_dir.exists():
        if local_dir.is_dir() and not any(local_dir.iterdir()):
            local_dir.rmdir()
        else:
            raise RuntimeError(f'{local_dir} exists; move it before linking persistent Drive storage')
    local_dir.symlink_to(drive_dir, target_is_directory=True)
print('Project:', Path.cwd())

## 2. Install project dependencies

Colab provides its own PyTorch/CUDA build. Install the project requirements without replacing Colab's PyTorch.

In [ ]:
!pip install -qU -r requirements.txt

In [ ]:
import torch, torchvision
print('PyTorch:', torch.__version__)
print('torchvision:', torchvision.__version__)
print('CUDA available:', torch.cuda.is_available())

## 3. Download the Figshare version 5 archive files

The downloader reads Figshare article version 5 metadata rather than relying on the old article-level `ndownloader` URL. It downloads the four image zip files into Drive, skips already complete files, and extracts them into the persistent `data/raw` folder.

In [ ]:
import pathlib
import requests
import zipfile

repo = pathlib.Path('/content/Deep-Hybrid-Learning-for-Brain-Tumor-Classification')
archives = STUDY_DIR / 'figshare_archives'
raw = repo / 'data' / 'raw'
archives.mkdir(parents=True, exist_ok=True)
raw.mkdir(parents=True, exist_ok=True)

url = 'https://api.figshare.com/v2/articles/1512427/versions/5'
response = requests.get(url, timeout=60)
response.raise_for_status()
article = response.json()

files = [
    f for f in article['files']
    if f['name'].startswith('brainTumorDataPublic_') and f['name'].endswith('.zip')
]
print('ZIP files:', [(f['name'], f['size']) for f in files])
assert len(files) == 4, 'Expected four dataset ZIP files; check the API file list above.'

for file_info in files:
    archive_path = archives / file_info['name']
    if not archive_path.exists() or archive_path.stat().st_size != file_info['size']:
        print('Downloading', file_info['name'])
        with requests.get(file_info['download_url'], stream=True, timeout=(30, 300)) as r:
            r.raise_for_status()
            with archive_path.open('wb') as out:
                for chunk in r.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        out.write(chunk)

    print('Extracting', file_info['name'])
    with zipfile.ZipFile(archive_path) as z:
        z.extractall(raw)

print('MAT files:', len(list(raw.rglob('*.mat'))))

## 4. Prepare a leakage-conscious train/validation/test split

The preparation script reads the MAT archives, creates PNGs and split manifests, grouping by patient ID where available and by exact file hashes. The split is generated before training augmentation. Re-running with the same inputs and seed recreates the split.

In [ ]:
# data/raw and data/processed are symlinks to the persistent Drive folders created above.
!python -m scripts.prepare_data --config configs/base.yaml

import pandas as pd
split_dir = PROCESSED_DIR / 'splits'
for split_name in ('train', 'val', 'test'):
    labels = pd.read_csv(split_dir / f'{split_name}.csv')['label']
    print(split_name, len(labels), labels.value_counts().to_dict())

## 5. Train the CNN baselines

Run each cell to completion before moving on. Every best checkpoint and training history is stored in Drive. Colab may disconnect during long runs; reconnect, rerun setup/data cells, then skip models whose checkpoint already exists or rerun a model to refresh its checkpoint.

In [ ]:
!python -m scripts.train --config configs/vgg16.yaml --checkpoint "$CHECKPOINT_DIR/vgg16.pt"

In [ ]:
!python -m scripts.train --config configs/mobilenet_v2.yaml --checkpoint "$CHECKPOINT_DIR/mobilenet_v2.pt"

In [ ]:
!python -m scripts.train --config configs/efficientnet_b3.yaml --checkpoint "$CHECKPOINT_DIR/efficientnet_b3.pt"

## 6. Train the four ViT variants

The configurations use ImageNet transfer learning and end-to-end fine-tuning, class-weighted loss, validation-based checkpoint selection, and early stopping. ViT-L has a smaller batch size to reduce GPU memory use. If a run runs out of memory, rerun that cell with batch size 1 or 2. These are project experiments; do not copy published values into the results as if they came from this run.

In [ ]:
!python -m scripts.train --config configs/vit_b_16.yaml --checkpoint "$CHECKPOINT_DIR/vit_b_16.pt"

In [ ]:
!python -m scripts.train --config configs/vit_b_32.yaml --checkpoint "$CHECKPOINT_DIR/vit_b_32.pt"

In [ ]:
!python -m scripts.train --config configs/vit_l_16.yaml --batch-size 2 --checkpoint "$CHECKPOINT_DIR/vit_l_16.pt"

In [ ]:
!python -m scripts.train --config configs/vit_l_32.yaml --batch-size 2 --checkpoint "$CHECKPOINT_DIR/vit_l_32.pt"

## 7. Evaluate the trained models on the held-out test split

Only evaluate a model after its checkpoint exists. Each report includes accuracy, balanced accuracy, macro and weighted precision/recall/F1, confusion matrix, and class-level metrics. The test set should be used once for final reporting, not hyperparameter tuning.

In [ ]:
!python -m scripts.evaluate --config configs/vgg16.yaml --checkpoint "$CHECKPOINT_DIR/vgg16.pt" --output "$REPORT_DIR/vgg16.json"

In [ ]:
!python -m scripts.evaluate --config configs/mobilenet_v2.yaml --checkpoint "$CHECKPOINT_DIR/mobilenet_v2.pt" --output "$REPORT_DIR/mobilenet_v2.json"

In [ ]:
!python -m scripts.evaluate --config configs/efficientnet_b3.yaml --checkpoint "$CHECKPOINT_DIR/efficientnet_b3.pt" --output "$REPORT_DIR/efficientnet_b3.json"

In [ ]:
!python -m scripts.evaluate --config configs/vit_b_16.yaml --checkpoint "$CHECKPOINT_DIR/vit_b_16.pt" --output "$REPORT_DIR/vit_b_16.json"

In [ ]:
!python -m scripts.evaluate --config configs/vit_b_32.yaml --checkpoint "$CHECKPOINT_DIR/vit_b_32.pt" --output "$REPORT_DIR/vit_b_32.json"

In [ ]:
!python -m scripts.evaluate --config configs/vit_l_16.yaml --checkpoint "$CHECKPOINT_DIR/vit_l_16.pt" --output "$REPORT_DIR/vit_l_16.json"

In [ ]:
!python -m scripts.evaluate --config configs/vit_l_32.yaml --checkpoint "$CHECKPOINT_DIR/vit_l_32.pt" --output "$REPORT_DIR/vit_l_32.json"

## 8. Compare results and build the ViT ensemble

The comparison CSV is ranked using validation metrics, with separate held-out test metrics alongside them. The four ViT checkpoints must all exist and use the same resolution before running the ensemble.

In [ ]:
!python -m scripts.compare_models --reports-dir "$REPORT_DIR" --output "$REPORT_DIR/comparison.csv"

from IPython.display import display
import pandas as pd
display(pd.read_csv(REPORT_DIR / 'comparison.csv'))

In [ ]:
!python -m scripts.ensemble --configs configs/vit_b_16.yaml configs/vit_b_32.yaml configs/vit_l_16.yaml configs/vit_l_32.yaml --checkpoints "$CHECKPOINT_DIR/vit_b_16.pt" "$CHECKPOINT_DIR/vit_b_32.pt" "$CHECKPOINT_DIR/vit_l_16.pt" "$CHECKPOINT_DIR/vit_l_32.pt" --output "$REPORT_DIR/vit_ensemble.json"

## 9. Resume and reporting checklist

- Check that every model has its own checkpoint and evaluation JSON in Drive.
- Report validation selection metrics separately from held-out test metrics.
- For imbalanced classes, include macro and per-class precision and recall, plus the confusion matrix; do not rank by accuracy alone.
- Report the split seed, model preprocessing, image resolution, batch size, optimizer settings, stopping rule, and Colab accelerator.
- Mark ViT results as pending until these training and evaluation cells finish. Published scores are literature context and are not this project's measurements.
- Download or retain the Drive folder `brain_tumor_study` with checkpoints, histories, reports, and split manifests for reproducibility.